# HVA (nuestro ansatz) vs Vector Loading (VL) de Haiqu

Compara la **preparación de estado por ansatz HVA** (lo que usa el pipeline GNN-HVA) contra el **Vector Loading de Haiqu** (síntesis de un estado a partir de su vector de amplitudes), en tres ángulos:

1. **Conceptual / metodológica** — qué problema resuelve cada uno.
2. **Cuantitativa (energía / fidelidad)** — qué tan cerca del ground state llega cada enfoque, en el mismo `(N, h)`.
3. **Recursos de circuito** — profundidad y gates de 2 qubits.

**Reutilización máxima (sin recalcular nada caro):** el theta de la GNN se lee de los `input_*.json` ya recuperados; el ground truth se toma del `ClassicalSolver` (cacheado en el notebook para no resolverlo dos veces); el circuito, el backend y el Hamiltoniano son los del repo. **No se entrena ninguna GNN.**

**Alcance:** simulación ideal (sin ruido). Los números de hardware real se citan de la recuperación previa. El **VL real de Haiqu no se ejecuta** (SDK cortado + repo aparte): se usa (a) el *VL ideal* = cargar el ground state exacto (fidelidad 1, energía E0) como referencia superior, y (b) un *proxy de recursos* vía la síntesis de state-preparation de Qiskit. Ambos etiquetados explícitamente.

## Trazabilidad de resultados

Los artefactos de este estudio se consolidan bajo `results/hva_vl_study/`, un subdirectorio por experimento (cada script referencia su propio archivo vía `source_script` + `result_path` en el JSON):

| Experimento | Script | Resultados |
|---|---|---|
| Expresividad del ansatz (2D) | `scripts/analysis/ansatz_expressivity_2d.py` | `results/hva_vl_study/expressivity/` |
| Caracterización VL | `scripts/analysis/vl_circuit_characterization.py` | `results/hva_vl_study/vl_characterization/` |
| Comparación de circuitos | `scripts/analysis/compare_circuits_square_n10_p2.py` | `results/hva_vl_study/circuit_comparison/` |
| h-sweep VL (frustrado) | `scripts/analysis/vl_h_sweep_frustrated.py` | `results/hva_vl_study/vl_h_sweep/` |
| h-sweep HVA+NNN (frustrado) | `scripts/analysis/hva_nnn_h_sweep_frustrated.py` | `results/hva_vl_study/hva_nnn_sweep/` |
| Fidelidad de preparación | `scripts/analysis/compare_hva_vl_mps_fidelity.py` | `results/hva_vl_study/state_prep_fidelity/` |
| Recursos HVA vs VL (χ) | `scripts/analysis/run_hva_vs_vl_resources.py` | `results/hva_vl_study/resources/` |

Helpers compartidos (rutas, `save_json` con trazabilidad, VQE robusto, transpile): `scripts/analysis/hva_vl_study_common.py`.


In [ ]:
import json
from functools import lru_cache
from pathlib import Path

import numpy as np

from qmbp_simulation import (
    ClassicalSolver,
    HamiltonianBuilder,
    HVACircuitBuilder,
    make_lattice,
)
from qmbp_simulation.execution import NoiselessBackend

TOPO, P, J, H = "heavy_hex", 1, 1.0, 1.0
DOWNLOADS = Path.home() / "Downloads"
backend = NoiselessBackend()
print("backend:", backend.name)

## 1. Comparación conceptual / metodológica

| Dimensión | HVA (bond-resolved, p=1) | Vector Loading de Haiqu |
|---|---|---|
| **Qué prepara** | Un estado dentro de la variedad del ansatz (limitado por p) | El estado objetivo arbitrario que le des |
| **Entrada** | Parámetros θ (19 para N=10) | El vector de amplitudes completo (2^N complejos) |
| **De dónde sale el estado** | Emergente; θ los predice la GNN | Hay que **conocer** el estado de antemano (ED/DMRG) |
| **Techo de expresividad** | **Sí** (no llega al autovalor exacto) | **No** (idealmente carga cualquier estado) |
| **Escala a N grande** | Sí (la GNN predice θ sin resolver el sistema) | **No** como solver: necesita el vector de 2^N amplitudes |
| **Profundidad** | Baja y controlada | Variable; crece con el entrelazamiento del estado |
| **Rol** | **Método de resolución** (aproxima lo desconocido) | **Método de carga** (materializa lo ya conocido) |

**Distinción de fondo:** son herramientas para propósitos opuestos. El HVA+GNN *descubre* una aproximación al ground state sin conocerlo; el VL *carga* un estado que ya resolviste clásicamente. Por eso el VL no compite con el HVA como solver: sirve como **referencia superior** (carga ideal) contra la cual medir cuánto pierde el HVA por su expresividad limitada.

In [ ]:
# --- Helpers reutilizando el repo. GT cacheado en-notebook (no resolver 2x). ---

def load_gnn_theta(n):
    d = json.load(open(DOWNLOADS / f"input_tfim_heavy_hex_n{n}_p1.json"))
    return np.asarray(d["parameters"], dtype=float)


@lru_cache(maxsize=8)
def ground_truth(n):
    lattice = make_lattice(TOPO, n, J=J, h=H)
    H_op = HamiltonianBuilder().build(lattice)
    gt = ClassicalSolver().solve(H_op, lattice)
    return lattice, H_op, gt


@lru_cache(maxsize=8)
def hva_circuit(n):
    lattice, _, _ = ground_truth(n)
    qc, _ = HVACircuitBuilder().create_bond_resolved(n, P, lattice)
    return qc


def energy_and_fidelity(n, theta):
    from qiskit.quantum_info import Statevector, state_fidelity

    _, H_op, gt = ground_truth(n)
    qc = hva_circuit(n)
    e = backend.evaluate(qc, H_op, np.asarray(theta))
    fid = None
    if gt.ground_state is not None:
        psi = backend.get_statevector(qc, np.asarray(theta))
        fid = float(state_fidelity(Statevector(psi), Statevector(gt.ground_state)))
    return e, fid, gt


print("helpers listos")

## 2. Comparación cuantitativa: energía y fidelidad

Para el mismo `(N, h)`:
- **HVA + θ GNN (ideal):** lo que corrió el pipeline, sin ruido.
- **HVA techo (θ óptimo clásico del ansatz):** el mejor punto del ansatz p=1 (de `/tmp/theta_ansatz_opt_n{N}.npy`; si falta, se omite).
- **VL ideal (Haiqu):** cargar el ground state exacto → E0, fidelidad 1. Referencia superior.
- **Referencia hardware:** energía medida en `ibm_pittsburgh` (recuperación previa).

In [ ]:
HW_ENERGY = {10: -11.5248, 20: -22.8241}  # ibm_pittsburgh, recuperado


def _fid(x):
    return f"{x:.4f}" if isinstance(x, float) else "n/a"


def quantitative_row(n):
    theta_gnn = load_gnn_theta(n)
    e_gnn, fid_gnn, gt = energy_and_fidelity(n, theta_gnn)
    e0 = float(gt.ground_energy)
    row = {
        "N": n, "E0": e0, "gap": float(gt.gap),
        "gnn_E": e_gnn, "gnn_dE": abs(e_gnn - e0), "gnn_fid": fid_gnn,
        "hw_E": HW_ENERGY.get(n),
        "hw_dE": (abs(HW_ENERGY[n] - e0) if n in HW_ENERGY else None),
    }
    p = Path(f"/tmp/theta_ansatz_opt_n{n}.npy")
    if p.exists():
        theta_ans = np.load(p)
        if theta_ans.size == hva_circuit(n).num_parameters:
            e_ans, fid_ans, _ = energy_and_fidelity(n, theta_ans)
            row["ceil_E"] = e_ans
            row["ceil_dE"] = abs(e_ans - e0)
            row["ceil_fid"] = fid_ans
    return row


rows = [quantitative_row(10)]
try:
    rows.append(quantitative_row(20))
except Exception as exc:  # noqa: BLE001
    print("N=20 omitido:", exc)

for r in rows:
    print("=" * 60)
    print(f"N={r['N']}  E0={r['E0']:+.4f}  gap={r['gap']:.4f}")
    print(f"  VL ideal (Haiqu)    : E={r['E0']:+.4f}  |dE|=0.0000  fid=1.000")
    if 'ceil_E' in r:
        print(f"  HVA techo (ansatz)  : E={r['ceil_E']:+.4f}  |dE|={r['ceil_dE']:.4f}  fid={_fid(r['ceil_fid'])}")
    print(f"  HVA + GNN (ideal)   : E={r['gnn_E']:+.4f}  |dE|={r['gnn_dE']:.4f}  fid={_fid(r['gnn_fid'])}")
    if r['hw_E'] is not None:
        print(f"  HVA + GNN (hardware): E={r['hw_E']:+.4f}  |dE|={r['hw_dE']:.4f}")
    if r['gnn_fid'] is None:
        print("  (N grande: el solver no devuelve el vector de estado -> fidelidad n/a)")

**Lectura:** el VL ideal alcanza E0 con fidelidad 1 por construcción (carga el estado exacto). El HVA tiene un techo (`HVA techo`) por debajo de E0 — esa brecha es el límite de expresividad del ansatz p=1, no del predictor. El θ de la GNN queda cerca del techo del ansatz, y el hardware añade la degradación por ruido. Jerarquía de energías (más negativo = mejor): `VL_ideal (E0) < HVA_techo < HVA_gnn_ideal < HVA_gnn_hardware`.

## 2b. Comparación reutilizable (script integrado)

La lógica de esta comparación vive en `scripts/analysis/compare_hva_vl_mps_fidelity.py`
(`compare_state_preparations`), reutilizable para cualquier lista de N. Predice θ con la
MPNN (sin VQE) para **HVA + GNN**, corre un VQE corto para el **techo del ansatz**, y usa
**VL ideal (Haiqu) = 1.0** por construcción. Solo produce filas para N ≤ límite statevector.


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / 'scripts').exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / 'scripts' / 'analysis'))  # for hva_vl_study_common
from scripts.analysis.compare_hva_vl_mps_fidelity import compare_state_preparations, format_rows

# Reutilizable: cambiá n_qubits/h a gusto. include_ceiling=False → sin VQE (más rápido).
rows = compare_state_preparations(
    topology=TOPO, n_qubits=[10, 16], h=H, p_layers=P,
    checkpoint='unifMPNN__heavy_hex_p1_signinv_fid_v1.pt',
    vqe_maxiter=150, vqe_restarts=3, include_ceiling=True,
)
print(format_rows(rows))
# (compare_state_preparations no persiste a disco; es cálculo en vivo.)

## 3. Recursos de circuito

Profundidad y gates de 2 qubits de:
- **Circuito HVA** (el que corrió), transpilado a base tipo IBM.
- **Proxy de VL:** síntesis genérica (`StatePreparation`) del **mismo ground state exacto**, misma base.

El proxy NO es el circuito comprimido real de Haiqu (que aplica su propia compresión, no accesible ahora): es una **cota superior** de referencia. Solo se calcula a N≤10, porque la síntesis genérica a N mayor es inviable (escala ~exponencial en gates 2q) — lo cual es, en sí, parte del punto.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation

BASIS = ["rz", "sx", "x", "cx"]
TWO_Q = {"cx", "cz", "ecr"}
MAX_VL_PROXY_N = 10  # sintesis generica solo tratable a N chico


def circuit_resources(n):
    _, _, gt = ground_truth(n)
    qc = hva_circuit(n)
    bound = qc.assign_parameters(load_gnn_theta(n))
    t_hva = transpile(bound, basis_gates=BASIS, optimization_level=2, seed_transpiler=42)
    res = {
        "N": n,
        "HVA_depth": t_hva.depth(),
        "HVA_2q": sum(v for op, v in t_hva.count_ops().items() if op in TWO_Q),
        "VLproxy_depth": None,
        "VLproxy_2q": None,
    }
    if gt.ground_state is not None and n <= MAX_VL_PROXY_N:
        psi = np.asarray(gt.ground_state, dtype=complex)
        psi = psi / np.linalg.norm(psi)
        vl = QuantumCircuit(n)
        vl.append(StatePreparation(psi), range(n))
        t_vl = transpile(vl, basis_gates=BASIS, optimization_level=2, seed_transpiler=42)
        res["VLproxy_depth"] = t_vl.depth()
        res["VLproxy_2q"] = sum(v for op, v in t_vl.count_ops().items() if op in TWO_Q)
    return res


res_rows = [circuit_resources(10)]
try:
    res_rows.append(circuit_resources(20))
except Exception as exc:  # noqa: BLE001
    print("N=20 recursos omitido:", exc)


def _cell(x):
    return str(x) if x is not None else "n/a (inviable)"


print(f"{'N':>3} | {'HVA depth':>9} {'HVA 2q':>7} | {'VLproxy depth':>14} {'VLproxy 2q':>14}")
print("-" * 60)
for r in res_rows:
    print(f"{r['N']:>3} | {r['HVA_depth']:>9} {r['HVA_2q']:>7} | "
          f"{_cell(r['VLproxy_depth']):>14} {_cell(r['VLproxy_2q']):>14}")

**Lectura de recursos:** el HVA es mucho más barato en gates de 2 qubits y profundidad que el proxy VL, porque solo aplica una capa de RZZ + RX, mientras que cargar un estado entrelazado arbitrario por síntesis genérica escala mal. Esto ilustra el trade-off central: el HVA sacrifica exactitud (techo de expresividad) por circuitos cortos ejecutables en NISQ; el VL alcanza el estado exacto pero con un circuito que, sin la compresión propietaria de Haiqu, es inviable a N grande.

> El VL real de Haiqu aplica compresión de estado que reduce estos números respecto al proxy genérico de Qiskit. El proxy es una **cota superior** de referencia, no la profundidad real de Haiqu.

## Conclusión

- **Energía/fidelidad:** el VL ideal domina (E0, fidelidad 1) porque no tiene techo de expresividad; el HVA queda limitado por el ansatz. Pero el VL solo puede hacerlo porque ya conocés el ground state — no es un método de descubrimiento.
- **Recursos:** el HVA es órdenes de magnitud más barato en profundidad y gates de 2 qubits.
- **Metodológico:** no son competidores directos. El HVA+GNN resuelve el problema de *encontrar* el estado a costo bajo y escalable; el VL *carga* un estado ya conocido a costo alto. El VL sirve como límite superior de calidad para cuantificar cuánto pierde el HVA por expresividad, y como técnica de preparación cuando el estado objetivo ya se conoce.

## 4. Visualizacion con el Haiqu SDK

Usa `haiqu.draw()` (devuelve una figura matplotlib) para dibujar, uno debajo del otro:
1. **Circuito HVA** (N=10, con el theta de la GNN) — el que corrio en `ibm_pittsburgh`.
2. **Proxy VL** — sintesis generica (`StatePreparation`) del ground state exacto, transpilada.

Y luego las **visualizaciones ricas** del `CircuitModel` REAL recuperado del dashboard
(solo lectura, sin re-ejecutar ni gastar QPU): matriz de correlacion, diversidad de
compuertas y radar. Estas son widgets HTML interactivos de Haiqu Lab — se renderizan
inline en Jupyter (no se exportan a PNG).

> Requiere `HAIQU_API_KEY` en el entorno y `haiqu-sdk >= 1.6.0`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from haiqu.sdk import haiqu as hq

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)
hq.login()  # usa HAIQU_API_KEY del entorno
print("Haiqu login OK")

### 4.1 HVA vs proxy VL (circuitos, uno debajo del otro)

In [ ]:
N = 10
_, _, gt10 = ground_truth(N)
qc_hva = hva_circuit(N).assign_parameters(load_gnn_theta(N))

# Proxy VL: sintesis del ground state exacto (mismo estado objetivo)
psi = np.asarray(gt10.ground_state, dtype=complex)
psi = psi / np.linalg.norm(psi)
qc_vl = QuantumCircuit(N)
qc_vl.append(StatePreparation(psi), range(N))
qc_vl_t = transpile(qc_vl, basis_gates=["rz", "sx", "x", "cx"],
                    optimization_level=2, seed_transpiler=42)

# hq.draw devuelve una figura matplotlib
fig_hva = hq.draw(qc_hva)
fig_hva.savefig(FIGDIR / "haiqu_hva_n10.png", dpi=110, bbox_inches="tight")
fig_vl = hq.draw(qc_vl_t)
fig_vl.savefig(FIGDIR / "haiqu_vlproxy_n10.png", dpi=110, bbox_inches="tight")
print(f"HVA: {qc_hva.num_qubits}q depth={qc_hva.depth()}  |  "
      f"VL proxy: {qc_vl_t.num_qubits}q depth={qc_vl_t.depth()}")
print("guardados: figures/haiqu_hva_n10.png, figures/haiqu_vlproxy_n10.png")

In [ ]:
# Mostrar los dos circuitos uno debajo del otro para comparacion visual
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 1, figsize=(14, 12))
for ax, png, title in [
    (axes[0], FIGDIR / "haiqu_hva_n10.png", "HVA bond-resolved p=1 (theta GNN) - el circuito que corrio"),
    (axes[1], FIGDIR / "haiqu_vlproxy_n10.png", "Proxy VL - sintesis del ground state exacto (transpilada)"),
]:
    ax.imshow(mpimg.imread(png))
    ax.set_title(title, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIGDIR / "haiqu_hva_vs_vlproxy_stack.png", dpi=110, bbox_inches="tight")
plt.show()
print("El HVA es una sola capa RZZ+RX (poco profundo). El proxy VL es enorme:\n"
      "visualmente ilustra el trade-off costo/expresividad.")

### 4.2 Visualizaciones ricas del CircuitModel REAL (recuperado del dashboard)

El circuito `circ-5346f90f-...` es el que ejecuto en `ibm_pittsburgh` (job `jb-4ccab22f`).
Lo recuperamos con `hq.get_circuit` (solo lectura) y usamos las visualizaciones del
`CircuitModel`. Son widgets HTML interactivos: se renderizan inline abajo.

In [ ]:
REAL_CIRCUIT_ID = "circ-5346f90f-a3e1-40c7-8ab9-6766f145f71b"  # job jb-4ccab22f, ibm_pittsburgh
cm = hq.get_circuit(REAL_CIRCUIT_ID)
print("CircuitModel:", cm.name, "| qubits:", cm.num_qubits, "| depth:", cm.depth)
# draw() del CircuitModel tambien devuelve figura matplotlib -> guardable
fig_real = cm.draw()
if hasattr(fig_real, "savefig"):
    fig_real.savefig(FIGDIR / "haiqu_real_circuit_n10.png", dpi=110, bbox_inches="tight")
    print("guardado: figures/haiqu_real_circuit_n10.png")

In [ ]:
# Widgets ricos (HTML interactivo, se muestran inline; no exportables a PNG)
cm.draw_gate_diversity()

In [ ]:
cm.draw_correlation_matrix()

In [ ]:
cm.draw_radar()

**Nota:** `draw_gate_diversity`, `draw_correlation_matrix` y `draw_radar` devuelven
widgets HTML (objeto `IPython.display.HTML`), no figuras matplotlib, por lo que se ven
inline en el notebook pero no se guardan como PNG. Los circuitos (`hq.draw` / `cm.draw`)
si devuelven figuras matplotlib y quedan guardados en `figures/`.

## 5. Caracterización del circuito VL REAL de Haiqu (depth / 2q-gates / fidelidad)

Ejecuta el **Vector Loading real** de Haiqu sobre el ground state exacto y recupera el
circuito generado para medir, más allá de su alta fidelidad, **qué profundidad y cuántos
gates de 2 qubits** cuesta — comparado con el HVA. Requiere `HAIQU_API_KEY` en el entorno.
Lógica en `scripts/analysis/vl_circuit_characterization.py` (`characterize_vl`).


In [ ]:
import os
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / 'scripts').exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / 'scripts' / 'analysis'))  # for hva_vl_study_common
from hva_vl_study_common import study_dir

from scripts.analysis.vl_circuit_characterization import characterize_vl, format_rows

assert os.environ.get('HAIQU_API_KEY'), 'export HAIQU_API_KEY antes de correr esta celda'
rows = characterize_vl(
    topology=TOPO, n_qubits=[6, 10], h_values=[2.5, 1.0], p_layers=P,
)
print(format_rows(rows))
print('→ Resultados:', study_dir('vl_characterization') / f'vl_char_{TOPO}.json')